# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
# Accessing `metadata` as an object
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review the available record sets, fields, and their IDs.

We will print all record sets and the fields within each, referencing all entities by their `@id`.

In [ ]:
# List all record sets (referenced by @id)
print("Available Record Sets (by @id):")
for rs in metadata.record_sets:
    print(f"- RecordSet @id: {rs.id}, name: {rs.name if hasattr(rs, 'name') else ''}")
    print("  Fields:")
    for field in rs.fields:
        print(f"    - Field @id: {field.id}\tname: {field.name}")

Let's display the first few records of a record set for an overview.
Replace `<rs_id>` below with an actual record set `@id` from the list above.

In [ ]:
# For demonstration, choose the main RecordSet (by @id)
# You may list metadata.record_sets to find the correct one
# Here we'll select the first one available as an example

first_record_set_id = metadata.record_sets[0].id if metadata.record_sets else None
if first_record_set_id:
    print(f"Sample records from RecordSet with @id: {first_record_set_id}")
    for i, record in enumerate(dataset.records(record_set=first_record_set_id)):
        print(record)
        if i > 2:
            break
else:
    print("No RecordSets available!")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. We'll extract all record sets.
Always reference record sets and fields by their `@id`.

In [ ]:
# Extract data from all record sets
record_sets_ids = [rs.id for rs in metadata.record_sets]
dataframes = {}

# Store all dataframes by record set @id
for record_set_id in record_sets_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df

# Display available columns from the first record set
if record_sets_ids:
    main_record_set_id = record_sets_ids[0]
    print(f"DataFrame Columns for RecordSet @id: {main_record_set_id}")
    print(dataframes[main_record_set_id].columns.tolist())
    display(dataframes[main_record_set_id].head())
else:
    print("No record sets to load data from.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

In this example:
  - We will identify a numeric field (e.g., 'coverage', 'peptide_count', 'MW_kDa', etc.) by its column name (which is the field `@id`/`name`)
  - Filter records based on a threshold
  - Normalize the field
  - Group by another field if it exists

**Replace the field variable assignments with real field @ids as needed for your dataset!**

In [ ]:
main_df = dataframes[main_record_set_id]
# Find a numeric field (by field @id/name)

# Try to find a column containing 'coverage', 'MW', 'count', or similar
potential_numeric_fields = [
    col for col in main_df.columns if any(
        x in col.lower() for x in ['coverage', 'mw', 'count', 'abundance', 'peptide', 'number']
    )
]

if not potential_numeric_fields:
    print("No obvious numeric fields found. Please inspect columns and set manually.")
    numeric_field = None
else:
    numeric_field = potential_numeric_fields[0]
    print(f"Using numeric field @id: {numeric_field}")

if numeric_field:
    # Cast to numeric in case it's type object (string)
    main_df[numeric_field] = pd.to_numeric(main_df[numeric_field], errors='coerce')

    # Set a threshold, e.g. mean or a fixed value
    threshold = main_df[numeric_field].mean() if main_df[numeric_field].notnull().any() else 0
    filtered_df = main_df[main_df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
    display(filtered_df.head())

    # Normalize the numeric column
    filtered_df[f"{numeric_field}_normalized"] = (
        filtered_df[numeric_field] - filtered_df[numeric_field].mean()
    ) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Group by another categorical field if present
    # Try to find a likely category field, e.g. 'sample', 'description', etc.
    categorical_fields = [
        col for col in main_df.columns
        if col != numeric_field and (main_df[col].dtype == 'object' or main_df[col].dtype.name == 'category')
    ]
    group_field = categorical_fields[0] if categorical_fields else None
    if group_field:
        print(f"Grouping by field @id: {group_field}")
        grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
        print(grouped_df.head())
else:
    print("No numeric field for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

Below we show histogram and (if available) boxplot/grouped bar of the numeric field. Adjust as needed.

In [ ]:
if numeric_field:
    plt.figure(figsize=(10, 4))
    sns.histplot(main_df[numeric_field].dropna(), bins=30)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.show()

    if group_field:
        plt.figure(figsize=(12,5))
        top_categories = filtered_df[group_field].value_counts().nlargest(10).index
        plot_df = filtered_df[filtered_df[group_field].isin(top_categories)]
        sns.boxplot(x=group_field, y=numeric_field, data=plot_df)
        plt.title(f"{numeric_field} grouped by {group_field} (top 10 categories)")
        plt.xticks(rotation=45)
        plt.show()
else:
    print("No numeric field available for visualization.")

## 6. Conclusion
In this notebook, we demonstrated how to explore the FAIR^2 dataset **Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells** using the `mlcroissant` library.

- We loaded the dataset metadata and reviewed record sets and their field `@id`s.
- Data was extracted from each record set as a pandas DataFrame.
- We performed basic exploratory data analysis, including filtering and normalization.
- Data visualizations highlighted the distribution and grouping of a key numeric field.

This workflow can be adapted to other Croissant datasets. For more complex analysis, consult specific field documentation provided in the schema, making reference to field `@id`s for reliable cross-dataset code.